# Module 2: Implement and Test a PyTorch-Based Classifier

> **Assignment (20 points):** Full PyTorch CNN training pipeline with questions + plots + predictions.

In [ ]:
import os, sys, pathlib, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
PROJECT_ROOT = pathlib.Path(os.path.abspath(os.getcwd()))
for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (p / 'images_dataSAT').is_dir():
        PROJECT_ROOT = p
        break
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
import os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

data_dir = './images_dataSAT'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

### Task 1 (Question): Why is random initialization useful for the model?

**Answer:** Random initialization breaks the *symmetry* between neurons — if all weights started with the same value, every neuron in a layer would compute identical gradients and be updated identically, so the network could never learn diverse features. Random small initial values also help gradients flow properly through the network (avoiding vanishing/exploding gradients) and allow different neurons to specialize on different patterns, which is essential for training deep models.

### Task 2: Create the training transformation pipeline `train_transform` using `transforms.Compose`

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(45),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print(train_transform)

### Task 3: Create the validation transformation pipeline `val_transform`

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print(val_transform)

## Datasets (70% train / 30% validation)

In [ ]:
train_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(data_dir, transform=val_transform)

train_size = int(0.7 * len(train_dataset))
val_size   = len(train_dataset) - train_size

train_dataset, _ = random_split(train_dataset, [train_size, val_size])
val_dataset,   _ = random_split(val_dataset, [val_size, train_size])

print('Train samples:', len(train_dataset), '| Val samples:', len(val_dataset))

### Task 4: Create `val_loader` for the validation dataset

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
print('Train batches:', len(train_loader), '| Val batches:', len(val_loader))

## CNN model definition

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = CNNClassifier()
print(model)

### Task 5 (Question): What is `tqdm` used for?

**Answer:** `tqdm` is a Python library that adds a progress bar to any loop. During training it shows the current iteration, elapsed time and the estimated remaining time (ETA), which makes monitoring long training epochs much easier.

### Task 6 (Question): Why are `train_loss`, `train_correct` and `train_total` set to 0 in every epoch?

**Answer:** These variables accumulate statistics for the *current* epoch only. If they were not reset, the values from previous epochs would carry over and the reported loss/accuracy would be cumulative and meaningless. Resetting them at the start of each epoch gives fresh per-epoch metrics.

### Task 7 (Question): Why do you need to use `torch.no_grad()` in the validation loop?

**Answer:** During validation we only need forward predictions — no gradients. `torch.no_grad()` disables autograd, so no computation graph is built or stored for backpropagation, which saves a lot of memory and speeds up evaluation.

### Task 8 (Question): What are two different metrics on which the model can be evaluated for best performance during training?

**Answer:** The two most common metrics are **validation accuracy** and **validation loss**. Alternative pairs such as **precision** and **recall** (or the **F1-score**) can also be used, especially for imbalanced datasets.

## Training loop (implements the reset pattern of Tasks 5–8)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 5
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

model = model.to(device)

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{EPOCHS}'):
        images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        train_correct += (preds == labels).sum().item()
        train_total += labels.numel()

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.numel()

    history['train_loss'].append(train_loss / train_total)
    history['val_loss'].append(val_loss / val_total)
    history['train_acc'].append(train_correct / train_total)
    history['val_acc'].append(val_correct / val_total)
    print(f'Epoch {epoch + 1}: train_loss={history["train_loss"][-1]:.4f} '
          f'val_loss={history["val_loss"][-1]:.4f} '
          f'train_acc={history["train_acc"][-1]:.4f} '
          f'val_acc={history["val_acc"][-1]:.4f}')

### Task 9: Plot the Model Loss from the training history of the model

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history['train_loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig('models/pytorch_cnn_loss.png')
plt.show()

### Task 10: Get `all_preds` (predictions) and `all_labels` (ground truth) from `val_loader`

In [ ]:
all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = torch.sigmoid(model(images))
        preds = (outputs > 0.5).long().squeeze(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print('Number of predictions:', len(all_preds))
print('First 10 predictions :', all_preds[:10])
print('First 10 labels      :', all_labels[:10])

## Evaluation metrics

In [ ]:
print('Accuracy :', round(accuracy_score(all_labels, all_preds), 4))
print('Precision:', round(precision_score(all_labels, all_preds), 4))
print('Recall   :', round(recall_score(all_labels, all_preds), 4))
print('F1 score :', round(f1_score(all_labels, all_preds), 4))
print('Confusion matrix:')
print(confusion_matrix(all_labels, all_preds))

os.makedirs('models', exist_ok=True)
torch.save(model.state_dict(), 'models/pytorch_cnn.pth')
print('Model saved -> models/pytorch_cnn.pth')